In [44]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')
class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"너는 사용자의 질문에 친철하고 정확하게 답변하는 시스템 입니다., Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,            
        )
        return response.choices[0].message.content
llm = OpenAILLM()    

In [45]:
import json
from  textwrap import dedent  #  들여쓰기 indent를 제거
query = '어제 삼성전자 주식종가 를 조회하고 해당 종가와 비슷한종목을 뉴스에서 검색해서 요약하고 오늘날씨에 맞는 외출복을 추천하고 데이트 장소 추천해'
def routerLLM(query):
    prompt = dedent(f"""
    사용자의 질문 의도를 분석하세요.

    질문에 포함된 의도가 여러 개이면
    반드시 모든 의도를 각각 분리하여 출력하세요.

    예를 들어:
    - 주식 + 뉴스 + 날씨
    - 뉴스 + 검색
    - 계산 + 주식

    처럼 복합 질문이면
    반드시 여러 개의 JSON 객체를 배열로 출력해야 합니다.

    question_type 별로 tool_query를 생성하세요.
    tool_query는 반드시 키워드중심으로 생성하세요 llm에 전달하는 문구가 아님을 명심해서 추론을하지말고 검색용 키워드로 매칭해주세요    
    뉴스는 api검색할수 있는 키워드중심으로,
    주식은 종목명만 추출하세요             
    날씨도 검색용 키워드로 추출하세요                               

    지원 가능한 question_type 예시:
    - 날씨
    - 뉴스
    - 주식
    - 검색
    - 계산
    - 지도

    [중요 규칙]
    - 질문에 포함된 모든 의도를 누락 없이 출력
    - 반드시 JSON 배열(Array) 형식으로 출력
    - 하나만 출력 금지
    - 설명문 금지
    - markdown 금지
    - ```json 금지

    [예시 입력]
    어제 삼성전자 종가 알려주고 관련 뉴스와 오늘 날씨도 알려줘

    [예시 출력]
    [
        {{
            "question_type": "주식",
            "tool_query": "삼성전자",
            "reason": "삼성전자 종가 요청"
        }},
        {{
            "question_type": "뉴스",
            "tool_query": "삼성전자 관련 최근 뉴스 검색",
            "reason": "관련 뉴스 요청"
        }},
        {{
            "question_type": "날씨",
            "tool_query": "오늘 날씨 조회",
            "reason": "날씨 요청"
        }}
    ]

    [질문]
    {query}

    [출력]
    반드시 유효한 JSON 배열만 출력하세요.
    """)

    result = llm.generate(prompt)
    json_result = json.loads(result)
    return json_result

In [46]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECETET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()    
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)        
        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [47]:
# tool excution
tool_results = []
def excute_tools(router_result):
    for row in router_result:
        query_type = row.get('question_type')
        print(f'tool : {query_type}')
        if query_type == '뉴스':        
            query = row.get('tool_query')
            news_result = search_naver_news(row.get('tool_query'))
            tool_results.append({
                'question_type' : '뉴스',
                'query' : query,
                'result' : news_result
            })
        # other tools
    return tool_results            

In [48]:
# final llm
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
def final_llm(user_query:str, tool_results:list[dict]):
    prompt = f'''
너는 다양한 도구의 결과를 종합해서 사용자 질문에 답변하는 ai assistant 입니다.

[사용자질문]
{user_query}

[도구실행결과]
{json.dumps(tool_results, ensure_ascii=False, indent=2)}

[지침]
- tool 결과를 기반으로 답변하세요
- 필요한 경우 여러 tool 결과를 종합하세요
- 지도 주식 날씨 검색 추천등 다양한 데이터가 포함될수 있습니다.
- tool 결과내에 있는 내용에 우선순위를 높게해서 해당 데이터기반으로 답변하세요
'''
    result = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages =[
                {"role":"system", "content":"너는 여러 tool결과를 조합해서 최종 답변을생성하는 ai agent 입니다."},
                {"role":"user", "content":prompt}
            ],
            temperature=0.3            
        )
    return result.choices[0].message.content

### 파이프라인
- 사용자 질문
- 질문의 의도를 분석하는 llm 호출
- 해당 tools을 호출해서 결과 수집
- 최종 llm에게 전달
- 최종 답변 생성

In [51]:
query = '붕괴사고에 대해서'
router_result = routerLLM(query)
tool_results = excute_tools(router_result)
final_result = final_llm(query,tool_results)
print(final_result)

tool : 뉴스
붕괴사고와 관련해, 제공된 뉴스 결과에서는 **서울 서소문 고가차도 붕괴사고**를 중심으로 수사·점검·수습 상황이 보도된 내용이 확인됩니다.

- **경찰 수사 진행(안전관리계획서 확보)**: 경찰이 서소문 고가차도 붕괴 사고 이후 **전담수사팀을 편성**하고, 서울시로부터 **관련 서류(안전관리계획서 등)**를 확보했으며, **감식 결과를 토대로 현장에서 규정과 절차가 제대로 지켜졌는지** 확인할 방침이라고 합니다.  
  (출처: https://n.news.naver.com/mnews/article/055/0001359504?sid=102)

- **정부 장관 현장 점검 및 원인 규명 주문**: 행정안전부 장관이 사고 현장을 찾아 **사고 원인에 대한 엄정 조사**와 **재발 방지 대책 마련**을 주문한 것으로 보도됐습니다.  
  (출처: https://www.joongangenews.com/news/articleView.html?idxno=521928)

- **서울시 브리핑(경위 및 수습대책)**: 서울시가 서소문 고가차도 붕괴 사고의 **발생 경위와 향후 계획/수습대책**에 대해 브리핑한 내용이 있습니다.  
  (출처: https://n.news.naver.com/mnews/article/421/0008968995?sid=102)

원하시면, **어떤 붕괴사고(예: 서소문 고가차도 외 다른 사고)**를 말씀하시는지 уточ해 주시면 그 사고에 맞춰 정리해 드릴게요.


In [52]:
# 1. planner / router : 사용자 요청을 분석하고 무슨작업이 필요한지 결정
# 2. tool excuter : router가 결정한 tool을 실제로 실행
# 3. memory / retrieval : 과거정보와 외부지식을 가져옮
# 4. LLM : 모든 tool결과와 memory를 종합해서 최종답변을 생성
# 5. final response : 사용자에게 보여줄 최종 출력 생성

### 모델을 허깅페이스계열의 모델로 변경

In [55]:
from huggingface_hub import notebook_login

os.environ.pop('HF_TOKEN', None)
notebook_login()

In [56]:
from huggingface_hub import get_token, whoami

HF_TOKEN = get_token()
if not HF_TOKEN:
    raise RuntimeError('Hugging Face 토큰이 저장되지 않았습니다. 로그인 셀을 다시 실행하세요.')

info = whoami(token=HF_TOKEN)
print('logged in user:', info.get('name'))
print('token prefix:', HF_TOKEN[:6] + '***')

logged in user: skn29teacher
token prefix: hf_apS***


In [6]:
%pip uninstall -y transformers

Found existing installation: transformers 5.9.0Note: you may need to restart the kernel to use updated packages.

Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0


In [7]:
%pip cache purge

Files removed: 18 (14.7 MB)
Directories removed: 0
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install transformers -U

   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ---------------------------------------- 10.8/10.8 MB 84.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [9]:
%pip install -r requirements_exaone.txt


   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------- ----------- 7.3/10.3 MB 64.9 MB/s eta 0:00:01
   ------------------------------ --------- 7.9/10.3 MB 20.3 MB/s eta 0:00:01
   ---------------------------------------- 10.3/10.3 MB 18.9 MB/s  0:00:00

  Attempting uninstall: transformers

    Found existing installation: transformers 5.9.0

   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
    Uninstalling transformers-5.9.0:
   -------------------- ------------------- 1/2 [transformers]
 

In [10]:
import os
# Windows 환경에서 trust_remote_code 사용 시 발생하는 signal.SIGALRM 에러 우회
os.environ["TRUST_REMOTE_CODE"] = "True"

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Choose your prompt
prompt = "Explain how wonderful you are"  # English example
prompt = "스스로를 자랑해 봐"       # Korean example

messages = [
    {"role": "system", 
     "content": "You are EXAONE model from LG AI Research, a helpful assistant."},
    {"role": "user", "content": prompt}
]
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

output = model.generate(
    input_ids.to("cuda"),
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=128,
    do_sample=False,
)
print(tokenizer.decode(output[0]))


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] You are using a model of type `exaone` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
[transformers] PreTrainedConfig got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 


AttributeError: 'PreTrainedConfig' object has no attribute 'max_position_embeddings'

### 모델 변경

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "너 스스로를 자랑해 봐"
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]